In [ ]:
import numpy as np
import pandas as pd
import yaml
from pandarallel import pandarallel
from tqdm import tqdm

pandarallel.initialize(progress_bar=True, nb_workers=16)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import data

In [ ]:
wos_classification = pd.read_parquet(dataset_config['path_processed'] + 'WOS/POST01_WOS_paper_US.parquet')
wos_classification

## Add Entity List info

In [ ]:
def locate_entity(entry):
    return False

In [ ]:
# Exceution time: 0 min
wos_classification['Entity_list'] = wos_classification['affiliationame'].parallel_apply(locate_entity)
wos_classification

In [ ]:
true_count = (wos_classification['Entity_list'] != False).sum()
print("Number of TRUE values:", true_count)

## Add variables

In [ ]:
wos_classification[wos_classification['type'] == 'f']

In [ ]:
wosid_with_firm = wos_classification[wos_classification['type'] == 'f']['wosid'].unique()
wosid_with_firm

In [ ]:
df_with_firm = wos_classification[wos_classification['wosid'].isin(wosid_with_firm)]
df_with_firm

In [ ]:
df_with_firm_dedup = df_with_firm.drop_duplicates(subset=['wosid', 'affiliationame'])[['wosid', 'affiliationame', 'type', 'Entity_list']]
df_with_firm_dedup

In [ ]:
# Export papers published by firms for further analysis
df_with_firm_dedup[['wosid', 'affiliationame']].to_parquet(dataset_config['path_processed'] + 'WOS/WOS_paperid_USfirm.parquet')

In [ ]:
for wosid, df in tqdm(df_with_firm_dedup.groupby(by='wosid')):
    if df['type'].apply(lambda x: 'u' in x).any():
        break

In [ ]:
results = []

for wosid, df in tqdm(df_with_firm_dedup.groupby(by='wosid')):
    df['with_uni'] = df['type'].apply(lambda x: 'u' in x).any()
    results.append(df)

In [ ]:
df_with_firm_uni = pd.concat(results)
df_with_firm_uni

In [ ]:
df_with_firm_uni['Entity_list'].value_counts()

### (1) Merge with paper info

In [ ]:
wos_paper_info = pd.read_parquet(dataset_config['path_processed'] + 'WOS/WOS_paper_level.parquet').drop_duplicates()
wos_paper_info.rename(columns={'pub year': 'year'}, inplace=True)
wos_paper_info

In [ ]:
df_with_firm_uni_paperinfo = pd.merge(df_with_firm_uni, wos_paper_info, on='wosid', how='left')
df_with_firm_uni_paperinfo

### (2) Merge with ISSN (delete the paper already in the CNKI)

In [ ]:
wos_cnki_overlap = pd.read_csv(dataset_config['path_processed'] + 'CNKI/overlap_cnkiwos_issn.csv', usecols=['journal_WOS'])
overlap_journals = wos_cnki_overlap['journal_WOS'] # Get the list of journals to remove
overlap_journals

In [ ]:
# Drop rows where the journal is in the overlap list
df_drop_overlap = df_with_firm_uni_paperinfo[
    ~df_with_firm_uni_paperinfo['journal'].isin(overlap_journals)
]
df_drop_overlap

### (3) Merge with JIF

In [ ]:
wos_jif = pd.read_parquet(dataset_config['path_processed'] + 'WOS/WOS_JIF.parquet')
wos_jif

In [ ]:
wos_all = pd.merge(df_drop_overlap, wos_jif, on='wosid', how='left')
wos_all

## Discriptive statistics: JIF

In [ ]:
# (1) Keep three columns
wos_filtered = wos_all[['wosid', 'year', 'impact_factor']].copy()

# (2) Drop duplicates
wos_filtered = wos_filtered.drop_duplicates()

# (3) Filter to 2010-2022
wos_period = wos_filtered[(wos_filtered['year'] >= 2000) & (wos_filtered['year'] <= 2022)]

# Overall mean
overall_mean = wos_period['impact_factor'].mean()
print("2010–2022年总体均值 JIF:", overall_mean)

# (4) Annual mean
yearly_means = wos_period.groupby('year')['impact_factor'].mean()
print("\n每年平均 JIF:")
print(yearly_means)

wos_period.to_csv(dataset_config['path_processed'] + 'WOS/WOS_USfirm_JIF.csv', index=False)


In [ ]:
stats = (
    wos_all["impact_factor"]
      .agg(max_value   = "max",
           min_value   = "min",
           mean_value  = "mean",
           median_value= "median")
      .round(1)
)

stats

## To firm-year level

In [ ]:
# 2 min
aff_stats = {
    "affiliationame": [],
    "year": [],
    "num_papers": [],
    "num_with_uni": [],
    "in_entity_list": [],
    "WOS_field": [],
    "num_papers_h": [],
    "num_with_uni_h": []
}

for aff_name, df in (
        wos_all.query("type == 'f'")
        .groupby("affiliationame")
):
    
    most_common_wos_area = df["WOS area"].mode()[0] if not df["WOS area"].mode().empty else None

    for year, sub_df in df.groupby("year"):
        aff_stats["affiliationame"].append(aff_name)
        aff_stats["year"].append(year)
        aff_stats["num_papers"].append(len(sub_df))
        aff_stats["num_with_uni"].append(sub_df["with_uni"].sum())
        aff_stats["in_entity_list"].append(sub_df["Entity_list"].iloc[0])
        aff_stats["WOS_field"].append(most_common_wos_area)

        hi = sub_df[sub_df["impact_factor"] > 3.1]
        aff_stats["num_papers_h"].append(len(hi))
        aff_stats["num_with_uni_h"].append(hi["with_uni"].sum())

In [ ]:
df_aff_stats = pd.DataFrame(aff_stats)
df_aff_stats

In [ ]:
# generate firm ID and field ID
df_aff_stats['firm_id'] = pd.factorize(df_aff_stats['affiliationame'])[0] + 1
df_aff_stats['field_id'] = pd.factorize(df_aff_stats['WOS_field'])[0] + 1
df_aff_stats

In [ ]:
df_aff_stats.to_csv(dataset_config['path_processed'] + 'WOS/WOS_USfirm_publication.csv', index=False)

## Statistics

In [ ]:
df_aff_stats[df_aff_stats['num_papers'] > 600]